In [1]:
!pip install pyarrow==2 awswrangler

In [2]:
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import random
import awswrangler as wr

# six month

In [3]:
sql = f"""
SELECT 
        dp.product_line as sub_product_line,   
        dp.henry_category_1,
        -- store
        fsoo.id_store,
        fsoo.store_name,
        fsoo.id_shop,
        CASE WHEN cast(fsoo.id_shop as bigint) = 1 THEN 'TH'
                                    WHEN cast(fsoo.id_shop as bigint) = 2 THEN 'SG'
                                    WHEN cast(fsoo.id_shop as bigint) = 5 THEN 'ID'
                                    WHEN cast(fsoo.id_shop as bigint) = 11 THEN 'MY'
                                    ELSE null
                                    END as id_shop_name
        -- gross revenue
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=0 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=6 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week1
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=7 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=13 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week2
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=14 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=20 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week3
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=21 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=27 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week4
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=28 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=34 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week5
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=35 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=41 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week6
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=42 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=48 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week7

        -- revenue
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=0 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=6 THEN revenue_usd ELSE 0 END) as revenue_usd_week1
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=7 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=13 THEN revenue_usd ELSE 0 END) as revenue_usd_week2
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=14 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=20 THEN revenue_usd ELSE 0 END) as revenue_usd_week3
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=21 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=27 THEN revenue_usd ELSE 0 END) as revenue_usd_week4
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=28 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=34 THEN revenue_usd ELSE 0 END) as revenue_usd_week5
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=35 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=41 THEN revenue_usd ELSE 0 END) as revenue_usd_week6
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=42 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=48 THEN revenue_usd ELSE 0 END) as revenue_usd_week7
        
        -- item discount
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=0 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=6 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week1
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=7 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=13 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week2
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=14 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=20 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week3
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=21 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=27 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week4
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=28 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=34 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week5
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=35 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=41 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week6
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=42 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=48 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week7
            
        -- voucher discount
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=0 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=6 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week1
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=7 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=13 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week2
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=14 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=20 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week3
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=21 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=27 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week4
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=28 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=34 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week5
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=35 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=41 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week6
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=42 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=48 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week7
        
        
           
   
        FROM dwh.dim_product dp


        -----------------------------------JOIN fsoo ---------------------------------------------
        --  net unit sold, gross unit sold, gross rev, rev by id_product_attribute by store  --
        LEFT JOIN
            (SELECT id_product_attribute,
                id_product,
                fso.id_store,
                date(fso.transaction_time) as transaction_date,
                fso.id_shop,
                fso.promotion_name as promotion_name,
                fso.transaction_type,
                CASE WHEN ds.store_name = 'CentralWorld' THEN 'Central World'
                     WHEN ds.store_name = 'Interchange21' THEN 'Interchange 21' 
                     WHEN ds.store_name = 'All Seasons' THEN 'All Seasons Place' 
                     ELSE ds.store_name END as store_name,
                COALESCE(SUM(CASE WHEN transaction_type ='Return' THEN -product_quantity ELSE product_quantity END),0) as net_units_sold,
                COALESCE(SUM(CASE WHEN transaction_type ='Sale' THEN product_quantity ELSE 0 END),0) as gross_units_sold,
                COALESCE(SUM(CASE WHEN transaction_type ='Sale' THEN gross_revenue_usd ELSE 0 END),0) as gross_revenue_usd,
                COALESCE(SUM(CASE WHEN transaction_type ='Sale' THEN revenue_usd ELSE 0 END),0) as revenue_usd
            FROM dwh.fact_sales_offline fso
            LEFT JOIN dwh.dim_store ds on fso.id_store = ds.id_store
            WHERE store_name NOT IN ('Pomelo Men pop up','Bangna Warehouse','Silom Complex','Werk Ari','Siam Center (closed)')
            GROUP BY 1,2,3,4,5,6,7,8) fsoo 
        ON fsoo.id_product_attribute = dp.id_product_attribute 


        WHERE date_released BETWEEN DATE_TRUNC('month', NOW() - INTERVAL '6' month) AND current_date
        AND fsoo.id_store NOT IN ('39-1')
        AND dp.parent_product_line != 'Free Gift'
        AND dp.henry_category_1 NOT IN ('Accessories','Bags','Bath&Body','Beverage Container','Cosmetics','Hair','Miscellaneous','Shoes','Skin Care','Stationery')
        AND dp.brand IN ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio','Blackdog BKK')
        AND dp.product_cost_usd_first_order_date > 0 
        AND dp.size !=  '33'
        AND dp.release_collection_name IS NOT NULL
        AND dp.fabric_custom_name IS NOT NULL
        AND dp.original_price_th_to_usd IS NOT NULL
        GROUP BY 1,2,3,4,5,6

"""

In [4]:
df = wr.athena.read_sql_query(sql, database="dwh")

In [5]:
df[
    (df["sub_product_line"] == "Basic")
    & (df["henry_category_1"] == "Bottoms")
    & (df["id_store"] == "251")
]

,sub_product_line,henry_category_1,id_store,store_name,id_shop,id_shop_name,gross_revenue_usd_week1,gross_revenue_usd_week2,gross_revenue_usd_week3,gross_revenue_usd_week4,...,item_discount_usd_week5,item_discount_usd_week6,item_discount_usd_week7,voucher_discount_usd_week1,voucher_discount_usd_week2,voucher_discount_usd_week3,voucher_discount_usd_week4,voucher_discount_usd_week5,voucher_discount_usd_week6,voucher_discount_usd_week7
8,Basic,Bottoms,251,EmQuartier,1,TH,270.566368,145.901063,229.71568,105.133845,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df[
    (df["sub_product_line"] == "Collection")
    & (df["henry_category_1"] == "Tops")
    & (df["id_store"] == "401")
]

,sub_product_line,henry_category_1,id_store,store_name,id_shop,id_shop_name,gross_revenue_usd_week1,gross_revenue_usd_week2,gross_revenue_usd_week3,gross_revenue_usd_week4,...,item_discount_usd_week5,item_discount_usd_week6,item_discount_usd_week7,voucher_discount_usd_week1,voucher_discount_usd_week2,voucher_discount_usd_week3,voucher_discount_usd_week4,voucher_discount_usd_week5,voucher_discount_usd_week6,voucher_discount_usd_week7
561,Collection,Tops,401,Seacon Bangkae,1,TH,602.700391,777.044794,535.05754,341.833402,...,0.0,0.0,0.0,5.540565,0.0,1.792867,0.0,0.0,0.0,0.0


In [7]:
df

,sub_product_line,henry_category_1,id_store,store_name,id_shop,id_shop_name,gross_revenue_usd_week1,gross_revenue_usd_week2,gross_revenue_usd_week3,gross_revenue_usd_week4,...,item_discount_usd_week5,item_discount_usd_week6,item_discount_usd_week7,voucher_discount_usd_week1,voucher_discount_usd_week2,voucher_discount_usd_week3,voucher_discount_usd_week4,voucher_discount_usd_week5,voucher_discount_usd_week6,voucher_discount_usd_week7
0,Basic,Tops,15,ID Central Park,5,ID,1236.282306,1426.722789,1947.135429,1825.925041,...,0.0,0.0,0.0,0.000000,0.000000,11.833077,61.479374,48.249065,58.556024,128.705666
1,Basic,Dresses,11,Icon Siam,1,TH,331.209962,373.746994,503.819182,222.667731,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Weekend,Tops,301,Central Phuket,1,TH,1552.357138,1923.713681,1136.546850,1707.572237,...,0.0,0.0,0.0,108.564826,39.053753,57.487312,41.354049,209.936627,106.316108,201.294329
3,Basic,Tops,371,Siam Center,1,TH,1464.137261,1219.926097,1043.845218,768.791173,...,0.0,0.0,0.0,30.633130,0.000000,0.000000,101.508284,49.807638,0.000000,0.000000
4,Collaboration,Outerwears,421,Central Festival Chiang Mai,1,TH,2917.164616,276.837249,71.325479,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
703,Collection,Dresses,441,Central Chidlom,1,TH,587.364740,767.894594,759.082390,680.550577,...,0.0,0.0,0.0,0.000000,9.976889,0.000000,0.000000,0.000000,0.000000,0.000000
704,Collection,Bottoms,381,Fashion Island,1,TH,509.199885,670.662513,283.357353,173.478669,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
705,Workwear,Outerwears,341,Central Ladprao,1,TH,3594.471577,2444.454561,1721.098808,961.139074,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
706,Weekend,Jumpsuits & Rompers,331,Zpell,1,TH,119.622955,30.045346,30.094442,60.026787,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [8]:
def calculate_day_no(row):
    if row["week_id"] == "week1":
        return 0
    if row["week_id"] == "week2":
        return 7
    if row["week_id"] == "week3":
        return 14
    if row["week_id"] == "week4":
        return 21
    if row["week_id"] == "week5":
        return 28
    if row["week_id"] == "week6":
        return 35
    if row["week_id"] == "week7":
        return 42

In [9]:
%%time
# unpivot 'gross_revenue_usd','revenue_usd','item_discount_usd','voucher_discount_usd'
unpivot_df = (
    pd.wide_to_long(
        df.reset_index(),
        stubnames=[
            "gross_revenue_usd",
            "revenue_usd",
            "item_discount_usd",
            "voucher_discount_usd",
        ],
        i="index",
        j="week_id",
        sep="_",
        suffix="\w+",
    )
    .reset_index(level=1)
    .reset_index(drop=True)
)

CPU times: user 57.4 ms, sys: 0 ns, total: 57.4 ms
Wall time: 58.7 ms


In [10]:
unpivot_df

,week_id,id_store,id_shop,store_name,henry_category_1,sub_product_line,id_shop_name,gross_revenue_usd,revenue_usd,item_discount_usd,voucher_discount_usd
0,week1,15,5,ID Central Park,Tops,Basic,ID,1236.282306,789.305051,0.0,0.000000
1,week1,11,1,Icon Siam,Dresses,Basic,TH,331.209962,309.482393,0.0,0.000000
2,week1,301,1,Central Phuket,Tops,Weekend,TH,1552.357138,1211.941566,0.0,108.564826
3,week1,371,1,Siam Center,Tops,Basic,TH,1464.137261,824.765573,0.0,30.633130
4,week1,421,1,Central Festival Chiang Mai,Outerwears,Collaboration,TH,2917.164616,2448.022629,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
4951,week7,441,1,Central Chidlom,Dresses,Collection,TH,39.866805,29.900104,0.0,0.000000
4952,week7,381,1,Fashion Island,Bottoms,Collection,TH,54.100994,49.969568,0.0,0.000000
4953,week7,341,1,Central Ladprao,Outerwears,Workwear,TH,24.381968,19.505575,0.0,0.000000
4954,week7,331,1,Zpell,Jumpsuits & Rompers,Weekend,TH,0.000000,0.000000,0.0,0.000000


In [25]:
discount = (
    unpivot_df.groupby(["sub_product_line", "henry_category_1", "week_id"])[
        "gross_revenue_usd", "revenue_usd", "item_discount_usd", "voucher_discount_usd"
    ]
    .mean()
    .reset_index()
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  if __name__ == '__main__':


In [26]:
discount

,sub_product_line,henry_category_1,week_id,gross_revenue_usd,revenue_usd,item_discount_usd,voucher_discount_usd
0,Basic,Bottoms,week1,25.239945,22.688757,0.000000,2.551188
1,Basic,Bottoms,week2,49.874774,38.935476,0.000000,10.936148
2,Basic,Bottoms,week3,61.223423,44.762876,0.000000,5.425995
3,Basic,Bottoms,week4,51.850509,37.127838,0.000000,3.378068
4,Basic,Bottoms,week5,54.916435,42.483128,0.000000,3.667500
...,...,...,...,...,...,...,...
191,Workwear,Tops,week3,251.061462,167.531989,6.908701,7.496527
192,Workwear,Tops,week4,212.165542,143.130104,6.740854,6.242429
193,Workwear,Tops,week5,126.155068,96.168196,0.000000,23.193406
194,Workwear,Tops,week6,65.401087,54.104874,0.000000,8.015263


In [27]:
# discount
discount["discount_utilization"] = np.where(
    discount["gross_revenue_usd"] == 0,
    0,
    1 - (discount["revenue_usd"] / discount["gross_revenue_usd"]),
)

discount["item_discount_percent"] = np.where(
    discount["gross_revenue_usd"] == 0,
    0,
    discount["item_discount_usd"] / discount["gross_revenue_usd"],
)

discount["voucher_discount_percent"] = np.where(
    discount["gross_revenue_usd"] == 0,
    0,
    discount["voucher_discount_usd"] / discount["gross_revenue_usd"],
)

In [28]:
discount

,sub_product_line,henry_category_1,week_id,gross_revenue_usd,revenue_usd,item_discount_usd,voucher_discount_usd,discount_utilization,item_discount_percent,voucher_discount_percent
0,Basic,Bottoms,week1,25.239945,22.688757,0.000000,2.551188,0.101077,0.000000,0.101077
1,Basic,Bottoms,week2,49.874774,38.935476,0.000000,10.936148,0.219335,0.000000,0.219272
2,Basic,Bottoms,week3,61.223423,44.762876,0.000000,5.425995,0.268860,0.000000,0.088626
3,Basic,Bottoms,week4,51.850509,37.127838,0.000000,3.378068,0.283945,0.000000,0.065150
4,Basic,Bottoms,week5,54.916435,42.483128,0.000000,3.667500,0.226404,0.000000,0.066783
...,...,...,...,...,...,...,...,...,...,...
191,Workwear,Tops,week3,251.061462,167.531989,6.908701,7.496527,0.332705,0.027518,0.029859
192,Workwear,Tops,week4,212.165542,143.130104,6.740854,6.242429,0.325385,0.031772,0.029422
193,Workwear,Tops,week5,126.155068,96.168196,0.000000,23.193406,0.237699,0.000000,0.183848
194,Workwear,Tops,week6,65.401087,54.104874,0.000000,8.015263,0.172722,0.000000,0.122555


In [29]:
del discount["gross_revenue_usd"]
del discount["revenue_usd"]
del discount["item_discount_usd"]
del discount["voucher_discount_usd"]

In [30]:
discount

,sub_product_line,henry_category_1,week_id,discount_utilization,item_discount_percent,voucher_discount_percent
0,Basic,Bottoms,week1,0.101077,0.000000,0.101077
1,Basic,Bottoms,week2,0.219335,0.000000,0.219272
2,Basic,Bottoms,week3,0.268860,0.000000,0.088626
3,Basic,Bottoms,week4,0.283945,0.000000,0.065150
4,Basic,Bottoms,week5,0.226404,0.000000,0.066783
...,...,...,...,...,...,...
191,Workwear,Tops,week3,0.332705,0.027518,0.029859
192,Workwear,Tops,week4,0.325385,0.031772,0.029422
193,Workwear,Tops,week5,0.237699,0.000000,0.183848
194,Workwear,Tops,week6,0.172722,0.000000,0.122555


In [31]:
discount.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/offline_clothing_v2/data/deploy_discount_offline.csv",
    index=False,
)